<a href="https://colab.research.google.com/github/Innovatewithapple/BioasqRAG/blob/main/BioASQRAG%CC%88Sparse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
import os
import torch

def setProductionSeed(isActive,seed=42):
  if isActive == True:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
      torch.cuda.manual_seed(seed)
      torch.cuda.manual_seed_all(seed)

      torch.backends.cudnn.deterministic = True
      torch.backends.cudnn.benchmark = False

setProductionSeed(True)

In [ ]:
#SParse
!pip install rank_bm25

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install transformers datasets

In [ ]:
import torch.nn as nn
from transformers import AutoModel,AutoTokenizer,AutoModelForCausalLM,AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset
from tqdm import tqdm
import torch.nn.functional as F
import re
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import sent_tokenize
import pandas as pd
import gc
from transformers import BitsAndBytesConfig

In [ ]:
EvaluationData = {}

In [ ]:
text_corpus = load_dataset('enelpol/rag-mini-bioasq','text-corpus')
question_answer_passage = load_dataset('enelpol/rag-mini-bioasq','question-answer-passages')

In [ ]:
text_corpus

In [ ]:
corpus_data = text_corpus['test']
corpus_data

In [ ]:
all_sentences = []

for data in corpus_data:
    sentences = sent_tokenize(data['passage'])

    for sentence in sentences:
        all_sentences.append(sentence)

with open("all_sentences.txt", "w", encoding="utf-8") as f:
    for sentence in all_sentences:
        f.write(sentence + "\n\n")

In [ ]:
#Do not call this unless you want to download csv of created chunks!!!!
rows = []

for data in corpus_data:

    passage_id = data['id']

    sentences = sent_tokenize(data['passage'])

    for sentence_idx, sentence in enumerate(sentences):

        rows.append({
            'passage_id': passage_id,
            'sentence_id': sentence_idx,
            'sentence': sentence
        })

df = pd.DataFrame(rows)

df.to_csv(
    "sentence_analysis_with_ids.csv",
    index=False
)

In [ ]:
def ExtractDataAndCreateChunks(chunksize, overlapsize):
    chunks = []
    for data in corpus_data:
        # Split into sentences
        sentences = re.split(r'(?<=[.!?])\s+', data['passage'])

        current_chunk = []
        current_length = 0

        for sentence in sentences:

            sentence_words = sentence.split()
            sentence_length = len(sentence_words)

            # If adding sentence exceeds chunk size
            if current_length + sentence_length > chunksize:

                chunk_text = " ".join(current_chunk)

                chunks.append({
                    'original_passage_id': data['id'],
                    'chunk_text': chunk_text
                })

                # overlap handling
                overlap_sentences = current_chunk[-overlapsize:]

                current_chunk = overlap_sentences
                current_length = sum(len(sentence.split()) for sentence in current_chunk)

            current_chunk.append(sentence)
            current_length += sentence_length

        # Add remaining chunk
        if current_chunk:

            chunk_text = " ".join(current_chunk)

            chunks.append({
                'original_passage_id': data['id'],
                'chunk_text': chunk_text
            })

    return chunks
knowledge_strings = ExtractDataAndCreateChunks(chunksize=100,overlapsize=1)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

**Sparse BM25**

In [ ]:
from rank_bm25 import BM25Okapi

In [ ]:
#sparse
from rank_bm25 import BM25Okapi
bm25_corpus = [
    chunk['chunk_text'].lower().split()
    for chunk in knowledge_strings
]

bm25 = BM25Okapi(bm25_corpus)

# **Encoder Model**

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
train_quesAnsPassage = question_answer_passage['train']
train_quesAnsPassage

In [ ]:
Reranking_token_Model = AutoTokenizer.from_pretrained("ncbi/MedCPT-Cross-Encoder")
rerank_model = AutoModelForSequenceClassification.from_pretrained("ncbi/MedCPT-Cross-Encoder")
rerank_model.eval()

In [ ]:

# del decoder_tokenizer
# gc.collect()
# torch.cuda.empty_cache()

In [ ]:
decoder_tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
decoder_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-3B-Instruct',
    dtype=torch.float16,
    device_map='auto'
)

In [ ]:
def GetFinalAnswer(input_ids_decoder,attention_mask):
  with torch.no_grad():
   with torch.amp.autocast('cuda'):
    output_ids = decoder_model.generate(
        input_ids_decoder,
        attention_mask=attention_mask,
        max_new_tokens=100,       # Gives it plenty of room to write a complete sentence [prev]
        do_sample=False,          # Enables natural conversational sampling [prev]
        top_k=10,                # Restricts the word pool to keep it highly focused [prev]
        temperature=0.2,         # Drops creativity near zero to lock onto context facts [prev]
        repetition_penalty=1.15, # Nudges it away from loops without breaking grammar rules [prev]
        no_repeat_ngram_size=0,  # Turned off: Disables the rigid, destructive phrase bans completely [prev]
        pad_token_id=decoder_tokenizer.eos_token_id
    )

   newTokens = output_ids[0,input_ids_decoder.shape[1]:]
   final_output = decoder_tokenizer.decode(newTokens,skip_special_token=True)
   print('FinalAnswer:', final_output)
   EvaluationData["predicted_answer"] = final_output
   return final_output


In [ ]:
def ManageExtractionPrompt(finalChunk):

    messages = [
        {
            "role": "system",
            "content": """
You are a biomedical findings extraction system.

Extract ONLY explicit biomedical findings and relationships stated in the CONTEXT.

Do NOT extract isolated entity names.

Focus on:
- biological relationships
- molecular mechanisms
- mutations
- biomarker associations
- epigenetic findings
- disease associations

Preserve biomedical entities exactly as written.

Return concise factual bullet points.

Do not infer unstated conclusions.
Do not introduce outside biomedical knowledge.
"""
        },

        {
            "role": "user",
            "content": f"""
CONTEXT:
{finalChunk}

EXTRACTED FINDINGS:
"""
        }
    ]

    text_prompt = decoder_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    input_ids_decoder = decoder_tokenizer.encode(
        text_prompt,
        return_tensors='pt'
    ).to(device)

    attention_mask = torch.ones_like(input_ids_decoder).to(device)

    return input_ids_decoder, attention_mask

In [ ]:
#Stage 2
def ManageAnswerPrompt(question, extractedFindings):

    messages = [
        {
            "role": "system",
            "content": """
You are a biomedical question answering system.

Answer the QUESTION using ONLY the provided FINDINGS.

Reuse biomedical entities exactly as written.

Do not introduce:
- new mutations
- new biomarkers
- new pathways
- new mechanisms

Do not infer unstated biological consequences.

Return a concise factual answer.
"""
        },

        {
            "role": "user",
            "content": f"""
QUESTION:
{question}

FINDINGS:
{extractedFindings}

ANSWER:
"""
        }
    ]

    text_prompt = decoder_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print("\nAnswer Prompt:\n", text_prompt)

    input_ids_decoder = decoder_tokenizer.encode(
        text_prompt,
        return_tensors='pt'
    ).to(device)

    attention_mask = torch.ones_like(input_ids_decoder).to(device)

    return input_ids_decoder, attention_mask

In [ ]:
def rerankedForm(question,retrieval_chunks,encoder_scores):
  pairs = []
  all_chunks = [i['chunk_text'] for i in retrieval_chunks]
  originalids = [i['original_passage_id'] for i in retrieval_chunks]
  for chunk in all_chunks:
    pairs.append([question,chunk])
  Reranking_tokenize_query = Reranking_token_Model(text=pairs,max_length=512,padding=True,truncation=True,return_tensors='pt')
  with torch.no_grad():
    with torch.amp.autocast('cuda'):
      reranking_model_output = rerank_model(Reranking_tokenize_query['input_ids'],Reranking_tokenize_query['attention_mask'])
      rerank_scores = reranking_model_output.logits.squeeze()

  # normalize encoder scores for top_k results
  # encoder_scores = top_scores[0]  # scores from your similarity computation
  # encoder_norm = (encoder_scores - encoder_scores.min()) / (encoder_scores.max() - encoder_scores.min())

  # # normalize reranker scores
  # rerank_norm = (rerank_scores - rerank_scores.min()) / (rerank_scores.max() - rerank_scores.min())

  # normalize retrieval scores safely
  if encoder_scores.max() == encoder_scores.min():
    encoder_norm = torch.zeros(len(encoder_scores))
  else:
    encoder_norm = (
        encoder_scores - encoder_scores.min()
    ) / (
        encoder_scores.max() - encoder_scores.min()
    )

  # normalize reranker scores safely
  if rerank_scores.max() == rerank_scores.min():
    rerank_norm = torch.zeros(len(rerank_scores))
  else:
    rerank_norm = (
        rerank_scores - rerank_scores.min()
    ) / (
        rerank_scores.max() - rerank_scores.min()
    )

  # combine both
  final_scores = torch.tensor(encoder_scores)
  sorted_indics = torch.argsort(final_scores, descending=True)
  # print('\n\nSorted_indics:',sorted_indics)
  print("\n===== RERANKED RESULTS =====\n")

  #Rerank the chunks again
  RerankedIds = []
  for idx in sorted_indics[:10]:
    RerankedIds.append(originalids[idx])
  final_chunks = [
    all_chunks[idx]
    for idx in sorted_indics[:5]
  ]
  print('\nRerankedIDS: ',RerankedIds)
  EvaluationData["retrieved_passage_ids"] = RerankedIds
  final_context = "\n\n".join(final_chunks)
  EvaluationData["retrieved_context"] = final_context
  # ===== STAGE 1 =====
  # Extraction
  input_ids_decoder, attention_mask = ManageExtractionPrompt(
    finalChunk=final_context
  )
  extracted_findings = GetFinalAnswer(
    input_ids_decoder=input_ids_decoder,
    attention_mask=attention_mask
  )
  print("\n===== EXTRACTED FINDINGS =====\n")
  print(extracted_findings)
  # ===== STAGE 2 =====
  # Final grounded answer
  input_ids_decoder, attention_mask = ManageAnswerPrompt(
    question=question,
    extractedFindings=extracted_findings
  )
  final_answer = GetFinalAnswer(
    input_ids_decoder=input_ids_decoder,
    attention_mask=attention_mask
  )
  print("\n===== FINAL ANSWER =====\n")
  print(final_answer)
  EvaluationData['predicted_answer'] = final_answer
  results = evaluate_rag(EvaluationData)
  EvaluationData['metrics'] = results
  print(results)
  # ManageDecoderModel(finalChunk=final_context,question=question)

In [ ]:
#Sparse
from rank_bm25 import BM25Okapi
import numpy as np

# ================================
# BUILD BM25 DATABASE
# ================================

bm25_corpus = [
    chunk['chunk_text'].lower().split()
    for chunk in knowledge_strings
]

bm25 = BM25Okapi(bm25_corpus)

# ================================
# QUESTION LOOP
# ================================

for quesIdx, i in enumerate(train_quesAnsPassage['question'][0:1], start=0):

    retrieval_chunks = []

    # =========================================
    # SPARSE BM25 RETRIEVAL
    # =========================================

    queryQ = i

    query_tokens = queryQ.lower().split()

    bm25_scores = bm25.get_scores(query_tokens)

    top_k = 100

    top_indices = np.argsort(bm25_scores)[::-1][:top_k]

    top_scores = [
        bm25_scores[idx]
        for idx in top_indices
    ]

    print(f'=====Matching=====')
    print(f'top_scores: {top_scores} | top_indexes: {top_indices}')
    print(f'{"="*50}')

    print(f'\n\nQuestion:{quesIdx}: {i}')

    print(
        f'\n\nExpected Answer: '
        f'{train_quesAnsPassage["answer"][quesIdx]}'
    )

    print(
        f'\n\nExpected AnswerID: '
        f'{train_quesAnsPassage["relevant_passage_ids"][quesIdx]}'
    )

    # =========================================
    # SAVE EVALUATION DATA
    # =========================================

    EvaluationData["question"] = i

    EvaluationData["Expected_Answer"] = (
        train_quesAnsPassage['answer'][quesIdx]
    )

    EvaluationData["Expected_Answer_ID"] = (
        train_quesAnsPassage['relevant_passage_ids'][quesIdx]
    )

    # =========================================
    # RETRIEVED CHUNKS
    # =========================================

    savedids = []

    for idx in top_indices:

        retrieval_chunks.append(
            knowledge_strings[idx]
        )

        savedids.append(
            knowledge_strings[idx]['original_passage_id']
        )

    print("Retrive-IDS: ", savedids)

    # =========================================
    # RERANKING
    # =========================================
    check_retrieval_match(expected_ids=EvaluationData["Expected_Answer_ID"],retrieved_ids=EvaluationData["retrieved_passage_ids"])
    rerankedForm(
        question=i,
        retrieval_chunks=retrieval_chunks,
        encoder_scores=torch.tensor(top_scores)
    )

=====Matching=====
top_scores: [np.float64(37.57431098269443), np.float64(36.51832621065066), np.float64(35.933706742234), np.float64(35.05294501122474), np.float64(34.7263262466983), np.float64(34.57179223924245), np.float64(34.36101828413658), np.float64(34.221142843577546), np.float64(33.90538650054557), np.float64(33.591131010717326), np.float64(33.31748369820075), np.float64(33.268254355061835), np.float64(33.13356472942086), np.float64(33.038274626873246), np.float64(33.02796697546642), np.float64(32.9735641137751), np.float64(32.82344894709407), np.float64(32.673594848602164), np.float64(32.66791723244269), np.float64(32.571238733348835), np.float64(32.302966945470644), np.float64(32.25328698023742), np.float64(32.13764616797075), np.float64(32.047322076759514), np.float64(32.03068268501464), np.float64(31.983736008024806), np.float64(31.945807104521666), np.float64(31.902946967560762), np.float64(31.84783908969596), np.float64(31.830155274524206), np.float64(31.790088756885908)

In [ ]:
#IDs Found or not
def check_retrieval_match(expected_ids, retrieved_ids):

    # remove duplicates while preserving order
    unique_retrieved = list(dict.fromkeys(retrieved_ids))

    # find matching ids
    matched_ids = []

    for doc_id in unique_retrieved:
        if doc_id in expected_ids:
            matched_ids.append(doc_id)

    total_expected = len(expected_ids)
    total_found = len(matched_ids)

    print("\n===== RETRIEVAL ANALYSIS =====\n")

    print("Expected IDs:")
    print(expected_ids)

    print("\nRetrieved IDs:")
    print(unique_retrieved)

    print("\nMatched IDs:")
    print(matched_ids)

    print(f"\nFound {total_found} out of {total_expected}")

    if total_found == total_expected:
        print("✅ ALL expected passages retrieved")

    elif total_found > 0:
        print("⚠️ PARTIAL retrieval success")

    else:
        print("❌ No expected passages retrieved")

    retrieval_percentage = (total_found / total_expected) * 100

    print(f"\nRetrieval Coverage: {retrieval_percentage:.2f}%")

    return {
        "matched_ids": matched_ids,
        "found_count": total_found,
        "total_expected": total_expected,
        "coverage_percent": retrieval_percentage
    }

In [ ]:
print("\n===== EVALUATION RESULTS =====\n")

for key, value in EvaluationData.items():

    if key != "metrics":

        print(f"{key} :\n{value}\n")

print("\n===== METRICS =====\n")

for metric_name, metric_value in EvaluationData['metrics'].items():

    print(f"{metric_name} : {metric_value}")

In [ ]:
# find and print the best chunk from this passage
for chunk in knowledge_strings:
    if chunk['original_passage_id'] == 23184418:
        print(chunk['chunk_text'])
        break

> Evaluation Metrics

In [ ]:
def evaluate_rag(EvaluationData):

    results = {}

    # =========================
    # RETRIEVAL METRICS
    # =========================

    results['hit@5'] = hit_at_k(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    results['recall@5'] = recall_at_k(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    results['precision@5'] = precision_at_k(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    results['mrr'] = reciprocal_rank(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids']
    )

    results['ndcg@5'] = calculate_ndcg(
        EvaluationData['Expected_Answer_ID'],
        EvaluationData['retrieved_passage_ids'],
        k=5
    )

    # =========================
    # GENERATION METRICS
    # =========================

    results['rouge'] = rouge_metric(
        EvaluationData['Expected_Answer'],
        EvaluationData['predicted_answer']
    )

    results['bertscore'] = bertscore_metric(
        EvaluationData['Expected_Answer'],
        EvaluationData['predicted_answer']
    )

    results['semantic_similarity'] = semantic_similarity(
        EvaluationData['Expected_Answer'],
        EvaluationData['predicted_answer']
    )

    # =========================
    # GROUNDEDNESS METRICS
    # =========================

    results['context_precision'] = context_precision(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['context_recall'] = context_recall(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['faithfulness'] = faithfulness(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['hallucination_score'] = hallucination_score(
        EvaluationData['retrieved_context'],
        EvaluationData['predicted_answer']
    )

    results['answer_relevance'] = answer_relevance(
        EvaluationData['question'],
        EvaluationData['predicted_answer']
    )

    return results

In [ ]:
!pip install sentence-transformers rouge-score bert-score
from transformers import pipeline
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score
from rouge_score import rouge_scorer
from sklearn.metrics import ndcg_score

sentence_Transformer_Model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
nli_pipeline = pipeline("text-classification",model="MoritzLaurer/deberta-v3-large-zeroshot-v2.0")

In [ ]:
#Hit@k: Did we retrive at least one correct document?
def hit_at_k(gold_ids, retrieved_ids, k=5):
    top_k = retrieved_ids[:k]
    for doc_id in top_k:
        if doc_id in gold_ids:
            return 1
    return 0

In [ ]:
#Recall@K: How many relevent documents were retrieved?
def recall_at_k(gold_ids, retrieved_ids, k=5):
    top_k = retrieved_ids[:k]
    relevant = 0
    for doc_id in top_k:
        if doc_id in gold_ids:
            relevant += 1
    return relevant / len(gold_ids)

In [ ]:
#Precision@K: How many retrieved documents are actually correct?
def precision_at_k(gold_ids, retrieved_ids, k=5):
    top_k = retrieved_ids[:k]
    relevant = 0
    for doc_id in top_k:
        if doc_id in gold_ids:
            relevant += 1
    return relevant / k

In [ ]:
#MRR (Mean Reciprocal Rank): How early was the FIRST correct document retrieved?
def reciprocal_rank(gold_ids, retrieved_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in gold_ids:
            return 1 / rank
    return 0

In [ ]:
#nDCG: Measures ranking quality.
def calculate_ndcg(gold_ids, retrieved_ids, k=5):
    relevance = []
    for doc_id in retrieved_ids[:k]:
        if doc_id in gold_ids:
            relevance.append(1)
        else:
            relevance.append(0)
    ideal = sorted(relevance, reverse=True)
    return ndcg_score([ideal], [relevance])

GENERATION METRICS

In [ ]:
#ROUGE
def rouge_metric(reference, prediction):
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rougeL'],
        use_stemmer=True
    )
    scores = scorer.score(reference, prediction)
    return {
        "rouge1": scores['rouge1'].fmeasure,
        "rougeL": scores['rougeL'].fmeasure
    }

In [ ]:
#BERTScore
def bertscore_metric(reference, prediction):
    P, R, F1 = score(
        [prediction],
        [reference],
        lang='en'
    )
    return {
        "precision": P.mean().item(),
        "recall": R.mean().item(),
        "f1": F1.mean().item()
    }

In [ ]:
#Semantic Similarity
def semantic_similarity(reference, prediction):
    embeddings = sentence_Transformer_Model.encode(
        [reference, prediction]
    )
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )
    return similarity[0][0]

GROUNDEDNESS METRICS

In [ ]:
#Context Precision: Did the answer actually use retrieved context?
def context_precision(retrieved_context,
                      predicted_answer):
    embeddings = sentence_Transformer_Model.encode(
        [retrieved_context, predicted_answer]
    )
    similarity = cosine_similarity(
        [embeddings[0]],
        [embeddings[1]]
    )
    return similarity[0][0]

In [ ]:
#Context Recall: Did the answer cover important retrieved facts?
def context_recall(retrieved_context,
                   predicted_answer):

    # split retrieved context into sentences
    context_sentences = [
        sent.strip()
        for sent in retrieved_context.split('.')
        if sent.strip()
    ]

    # embed predicted answer once
    answer_embedding = sentence_Transformer_Model.encode(
        [predicted_answer]
    )

    similarities = []

    # compare every context sentence against answer
    for sentence in context_sentences:

        sentence_embedding = sentence_Transformer_Model.encode(
            [sentence]
        )

        similarity = cosine_similarity(
            sentence_embedding,
            answer_embedding
        )[0][0]

        similarities.append(similarity)

    # average coverage score
    return np.mean(similarities)

In [ ]:
#Faithfulness (NLI-Based): measures hallucination
def faithfulness(retrieved_context,predicted_answer):
    result = nli_pipeline( f"{retrieved_context} [SEP] {predicted_answer}")
    label = result[0]['label'].lower()
    score = result[0]['score']
    if label == "entailment":
        return score
    elif label == "contradiction":
        return 1 - score
    else:
        return 0.5

In [ ]:
#Hallucination Score
def hallucination_score(retrieved_context,predicted_answer):
    faithful_score = faithfulness(retrieved_context,predicted_answer)
    return 1 - faithful_score

In [ ]:
#Answer Relevance
def answer_relevance(question,predicted_answer):
    embeddings = sentence_Transformer_Model.encode([question, predicted_answer])
    similarity = cosine_similarity([embeddings[0]],[embeddings[1]])
    return similarity[0][0]